In [1]:
import pandas as pd
import fastf1
from pathlib import Path

fastf1.Cache.enable_cache("../cache")
ROOT_DIR = Path.cwd().parent.resolve()
ROOT_DIR

WindowsPath('D:/Projects/formula1-data-analysis')

In [8]:
year = 2025

raw_folder = (
    ROOT_DIR
    / "data"
    / "raw"
    / str(year)
)

races = [
    race
    for race in raw_folder.iterdir()
    if race.is_dir()
]

print(f"Found {len(races)} races")

all_degradation_metrics = []

for race in races:

    race_name = race.name

    print(f"Processing {race_name}")

    laps_file = race / "laps.csv"

    if not laps_file.exists():

        print(
            f"No laps file found "
            f"for {race_name}"
        )

        continue

    laps = pd.read_csv(
        laps_file
    )

    laps = laps[
        laps["LapTime"].notna()
    ].copy()

    if laps.empty:

        continue

    laps["LapTime"] = (
        pd.to_timedelta(
            laps["LapTime"]
        )
        .dt.total_seconds()
    )

    stints = laps.groupby(
        [
            "Driver",
            "Stint",
            "Compound"
        ]
    )

    for (
        driver,
        stint,
        compound
    ), stint_laps in stints:

        if len(stint_laps) < 6:

            continue

        stint_laps = (
            stint_laps
            .sort_values(
                "LapNumber"
            )
        )

        avg_pace = (
            stint_laps["LapTime"]
            .mean()
        )

        first_three_avg = (
            stint_laps
            .head(3)["LapTime"]
            .mean()
        )

        last_three_avg = (
            stint_laps
            .tail(3)["LapTime"]
            .mean()
        )

        pace_dropoff = (
            last_three_avg
            - first_three_avg
        )

        degradation_rate = (
            pace_dropoff
            / len(stint_laps)
        )

        all_degradation_metrics.append(
            {
                "Race": race_name,
                "Driver": driver,
                "Compound": compound,
                "Stint": stint,
                "StintLength":
                    len(stint_laps),
                "AvgPace":
                    avg_pace,
                "FirstThreeAvg":
                    first_three_avg,
                "LastThreeAvg":
                    last_three_avg,
                "PaceDropoff":
                    pace_dropoff,
                "DegradationRate":
                    degradation_rate
            }
        )

tire_degradation = pd.DataFrame(
    all_degradation_metrics
)

print(
    f"Created "
    f"{len(tire_degradation)} "
    f"records"
)

tire_degradation.head()
    

Found 24 races
Processing abu_dhabi
Processing australian
Processing austrian
Processing azerbaijan
Processing bahrain
Processing belgian
Processing british
Processing canadian
Processing chinese
Processing dutch
Processing emilia_romagna
Processing hungarian
Processing italian
Processing japanese
Processing las_vegas
Processing mexico_city
Processing miami
Processing monaco
Processing qatar
Processing saudi_arabian
Processing singapore
Processing spanish
Processing são_paulo
Processing united_states
Created 1114 records


,Race,Driver,Compound,Stint,StintLength,AvgPace,FirstThreeAvg,LastThreeAvg,PaceDropoff,DegradationRate
0,abu_dhabi,ALB,SOFT,1.0,8,91.937000,93.897000,90.872333,-3.024667,-0.378083
1,abu_dhabi,ALB,HARD,2.0,25,90.987880,96.180333,90.880000,-5.300333,-0.212013
2,abu_dhabi,ALB,MEDIUM,3.0,25,89.704120,95.301333,89.140000,-6.161333,-0.246453
3,abu_dhabi,ALO,MEDIUM,1.0,16,90.572563,91.811667,90.677000,-1.134667,-0.070917
4,abu_dhabi,ALO,HARD,2.0,42,90.139762,97.117667,89.046333,-8.071333,-0.192175
